# PS S6E5 - 053 RealMLP watchlist updated Optuna search min

Experiment ID: `exp_20260525_053_realmlp_watchlist_updated_optuna_search_min`

Purpose:
- Use `exp_20260525_051_realmlp_watchlist_updated_shared001v2_min` as the fixed base.
- Keep the 051 feature engineering, folds, original-row concat, and target encoding policy unchanged.
- Run narrow Optuna search only for RealMLP `lr`, `wd`, and `p_drop`.
- Keep `n_epochs=5` fixed in this first search stage.
- Use a lighter search setting with `n_ens=8` to reduce runtime.

Notes:
- This is an Optuna search notebook, not a formal OOF/PRED/submission notebook.
- The formal refit should be done as 054 using `best_params_053.json`.
- Do not judge final adoption from 053 alone; judge after 054 single model and Add054 blend.


In [1]:
%%time
!pip install -q pytabkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 8.2 MB/s eta 0:00:00
CPU times: user 58 ms, sys: 11.4 ms, total: 69.4 ms
Wall time: 5.51 s


In [2]:
import os
import json
import random
import warnings
import pickle
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from colorama import Fore, Style
from importlib.metadata import version
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import KBinsDiscretizer, TargetEncoder

import optuna
import torch
import pytabkit
from pytabkit import RealMLP_TD_Classifier

warnings.filterwarnings('ignore')
print("PyTorch  version:", torch.__version__)
print("PyTabKit version:", version("pytabkit"))

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
print("Optuna version:", optuna.__version__)


PyTorch  version: 2.10.0+cu128
PyTabKit version: 1.7.3
Optuna version: 4.8.0


In [3]:
# Experiment management config
EXP_ID = "exp_20260525_053_realmlp_watchlist_updated_optuna_search_min"
COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
METRIC = "AUC"
CREATED_AT = "2026-05-25"

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)


EXP_ID: exp_20260525_053_realmlp_watchlist_updated_optuna_search_min
OUTDIR: /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min


## Load Data

In [4]:
# Data paths
COMP_DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e5")
ORIG_DATA_PATH = Path("/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv")

train = pd.read_csv(COMP_DATA_DIR / "train.csv")
test = pd.read_csv(COMP_DATA_DIR / "test.csv")
orig = pd.read_csv(ORIG_DATA_PATH)

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Orig shape :", orig.shape)


Train shape: (439140, 16)
Test shape : (188165, 15)
Orig shape : (101371, 16)


## Preprocess Features

In [5]:
%%time
ID = 'id'
TARGET = 'PitNextLap'
orig = orig.drop(['Normalized_TyreLife'], axis=1, errors='ignore')
y_orig = orig[TARGET]; orig = orig.drop([TARGET], axis=1)
X = train.drop([ID, TARGET], axis=1); train_id = train[ID]
y = train[TARGET]
X_test = test.drop([ID], axis=1); test_id = test[ID]
del train, test
print("X      init shape:", X.shape)
print("X_test init shape:", X_test.shape)
print("orig   init shape:", orig.shape, "\n")

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()
print("init len(cat_cols):", len(cat_cols))
print("init len(num_cols):", len(num_cols), "\n")

category_map = {}
important_combos = [
    ('Race', 'Compound'),
    ('Race', 'Year'),
]
def feature_engineering(df, fit=False):
    # Arithmetic interaction
    df['_LapNumber_/_RaceProgress'] = (df['LapNumber'] / (df['RaceProgress'] + 1e-6)).astype('float32')
    df['_TyreLife_/_LapNumber'] = (df['TyreLife'] / df['LapNumber'].clip(lower=1)).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation'] = (df['LapTime (s)'] * df['Cumulative_Degradation']).astype('float32')
    df['_LapTime (s)_*_Cumulative_Degradation_abs'] = (df['LapTime (s)'] * df['Cumulative_Degradation'].abs()).astype('float32')
    df['_LapTime (s)_/_Cumulative_Degradation_abs'] = (df['LapTime (s)'] / (df['Cumulative_Degradation'].abs() + 1e-6)).astype('float32')

    # Categorize numericals
    for col in num_cols + ['_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber']:
        cat_name = f"{col}_cat_" if col in num_cols else f"{col[1:]}_cat_"
        if fit:
            codes, uniques = np.floor(df[col]).factorize()
            category_map[col] = uniques
        else:
            uniques = category_map[col]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = np.floor(df[col]).map(code_map).fillna(-1).astype('int32')
        df[cat_name] = codes
        df[cat_name] = df[cat_name].astype(str)

    # Count encoding
    for col in cat_cols + ['Year_cat_', 'PitStop_cat_']:
        count_name = f"_{col}_count" if col in cat_cols else f"_{col[:-1]}_count"
        if fit:
            count_map = df[col].value_counts()
            category_map[count_name] = count_map
        else:
            count_map = category_map[count_name]
        df[count_name] = df[col].map(count_map).fillna(0).astype('int32')

    # Discretize numericals
    bin_config = {'RaceProgress': [200], 'LapTime (s)': [7]}
    for col, bins_list in bin_config.items():
        for n_bins in bins_list:
            for strategy in ['quantile']:
                bin_name = f"{col}_{n_bins}_{strategy}_bin_"
                if fit:
                    kb = KBinsDiscretizer(
                        n_bins=n_bins,
                        encode='ordinal',
                        strategy=strategy,
                        subsample=None
                    )
                    binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                    category_map[bin_name] = kb
                else:
                    kb = category_map[bin_name]
                    binned = kb.transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = binned
                df[bin_name] = df[bin_name].astype(str)

    # Create interaction categories
    combo_names = []
    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_names.append(combo_name)
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]:
            combo_series = combo_series + '_' + df[col].astype(str)
        if fit:
            codes, uniques = pd.factorize(combo_series, sort=False)
            category_map[combo_name] = uniques
        else:
            uniques = category_map[combo_name]
            code_map = {cat: i for i, cat in enumerate(uniques)}
            codes = combo_series.map(code_map).fillna(-1).astype('int32')
        df[combo_name] = codes
        df[combo_name] = df[combo_name].astype(str)   

    new_cat_cols = [col for col in df.columns if col.endswith('_')]
    new_num_cols = [col for col in df.columns if col.startswith('_')]
    return df, new_cat_cols, new_num_cols, combo_names

X, new_cat_cols, new_num_cols, combo_names = feature_engineering(X, fit=True)
X_test, new_cat_cols, new_num_cols, combo_names = feature_engineering(X_test, fit=False)
orig, new_cat_cols, new_num_cols, combo_names = feature_engineering(orig, fit=False)
cat_cols += new_cat_cols; num_cols += new_num_cols
print("len(new_cat_cols):", len(new_cat_cols))
print("len(new_num_cols):", len(new_num_cols), "\n")

print("prep len(cat_cols):", len(cat_cols))
print("prep len(num_cols):", len(num_cols), "\n")
print("X      prep shape:", X.shape)
print("X_test prep shape:", X_test.shape)
print("orig   prep shape:", orig.shape, "\n")
# Save base feature columns after source feature engineering, before fold-wise TE columns.
base_feature_columns = X.columns.tolist()
pd.DataFrame({"feature": base_feature_columns}).to_csv(OUTDIR / "feature_columns_base_before_te.csv", index=False)
print("Saved base feature columns:", OUTDIR / "feature_columns_base_before_te.csv")


X      init shape: (439140, 14)
X_test init shape: (188165, 14)
orig   init shape: (101371, 14) 

init len(cat_cols): 3
init len(num_cols): 11 

len(new_cat_cols): 17
len(new_num_cols): 10 

prep len(cat_cols): 20
prep len(num_cols): 21 

X      prep shape: (439140, 41)
X_test prep shape: (188165, 41)
orig   prep shape: (101371, 41) 

Saved base feature columns: /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/feature_columns_base_before_te.csv
CPU times: user 3.44 s, sys: 407 ms, total: 3.85 s
Wall time: 3.86 s


## Config

In [6]:
class CFG:
    FOLDS = 5
    SEED = 42
    TE = True

    # Optuna search control
    N_TRIALS = 20
    TIMEOUT_SEC = 60 * 60 * 10
    OPTUNA_SEED = 42

    # Search-stage RealMLP runtime. 054 refit should return to n_ens=20.
    SEARCH_N_ENS = 8
    SEARCH_N_EPOCHS = 5

# 051 baseline params. Optuna only searches lr / wd / p_drop.
BASE_PARAMS = {
    'random_state': 42,
    'verbosity': 2,
    'val_metric_name': '1-auc_ovr',

    'n_ens': CFG.SEARCH_N_ENS,
    'n_epochs': CFG.SEARCH_N_EPOCHS,
    'batch_size': 256,
    'use_early_stopping': False,
    'early_stopping_additive_patience': 10,
    'early_stopping_multiplicative_patience': 1,

    'lr': 0.019,
    'wd': 0.01,
    'sq_mom': 0.99,
    'lr_sched': 'lin_cos_log_15',
    'first_layer_lr_factor': 0.25,

    'embedding_size': 6,
    'max_one_hot_cat_size': 18,
    'hidden_sizes': [512, 256, 128],
    'act': 'silu',
    'p_drop': 0.05,
    'p_drop_sched': 'invsqrtp1e-3',

    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_act_name': 'gelu',
    'plr_lr_factor': 0.1151,
    'plr_sigma': 2.33,

    'ls_eps': 0.01,
    'ls_eps_sched': 'sqrt_cos',

    'add_front_scale': False,
    'bias_init_mode': 'neg-uniform-dynamic-2',
    'tfms': ['one_hot', 'median_center', 'robust_scale',
             'smooth_clip', 'embedding', 'l2_normalize'],
}

SEARCH_SPACE = {
    "lr": {"type": "float_log", "low": 0.012, "high": 0.026},
    "wd": {"type": "float_log", "low": 0.004, "high": 0.025},
    "p_drop": {"type": "float", "low": 0.03, "high": 0.08},
}

REFERENCE = {
    "baseline_051_cv_auc": 0.9540932672601146,
    "baseline_051_public_lb": 0.95407,
    "current_best_attack_052_nm_cv_auc": 0.9550261129800026,
    "current_best_attack_052_nm_public_lb": 0.95435,
    "current_best_stable_052_logreg_cv_auc": 0.9550030438849962,
    "current_best_stable_052_logreg_public_lb": 0.95432,
}

print("BASE_PARAMS searched keys only:", {k: BASE_PARAMS[k] for k in ["lr", "wd", "p_drop", "n_ens", "n_epochs"]})
print("SEARCH_SPACE:", SEARCH_SPACE)


BASE_PARAMS searched keys only: {'lr': 0.019, 'wd': 0.01, 'p_drop': 0.05, 'n_ens': 8, 'n_epochs': 5}
SEARCH_SPACE: {'lr': {'type': 'float_log', 'low': 0.012, 'high': 0.026}, 'wd': {'type': 'float_log', 'low': 0.004, 'high': 0.025}, 'p_drop': {'type': 'float', 'low': 0.03, 'high': 0.08}}


## Optuna Search

Search only `lr`, `wd`, and `p_drop` while keeping `n_epochs=5` fixed.  
This notebook does not create formal OOF/PRED/submission. 054 should refit the best parameters with `n_ens=20`.


In [7]:
%%time

def make_trial_params(trial):
    params = BASE_PARAMS.copy()
    params.update({
        "lr": trial.suggest_float("lr", SEARCH_SPACE["lr"]["low"], SEARCH_SPACE["lr"]["high"], log=True),
        "wd": trial.suggest_float("wd", SEARCH_SPACE["wd"]["low"], SEARCH_SPACE["wd"]["high"], log=True),
        "p_drop": trial.suggest_float("p_drop", SEARCH_SPACE["p_drop"]["low"], SEARCH_SPACE["p_drop"]["high"]),
    })
    return params


def run_cv_for_params(params, trial=None):
    skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
    oof_preds = np.zeros(len(X), dtype=np.float32)
    fold_scores = []
    fold_details = []
    final_feature_columns = None

    for fold, ((tr_idx, val_idx), (or_tr_idx, or_val_idx)) in enumerate(
              zip(skf.split(X, y), skf.split(orig, y_orig)), 1):
        X_tr = X.iloc[tr_idx].copy()
        orig_tr = orig.iloc[or_tr_idx].copy()
        X_tr = pd.concat([X_tr, orig_tr], axis=0).reset_index(drop=True)
        y_tr = pd.concat([y.iloc[tr_idx], y_orig.iloc[or_tr_idx]], axis=0).reset_index(drop=True)
        X_val = X.iloc[val_idx].copy()
        y_val = y.iloc[val_idx]

        # Target encoding. Kept as source notebook behavior.
        if CFG.TE:
            te_cols = combo_names
            TE = TargetEncoder(cv=CFG.FOLDS, smooth='auto', shuffle=True, random_state=CFG.SEED)
            tr_enc = TE.fit_transform(X_tr[te_cols], y_tr)
            val_enc = TE.transform(X_val[te_cols])

            te_names = [f"_{col}TE" for col in te_cols]
            X_tr[te_names] = tr_enc
            X_val[te_names] = val_enc
        else:
            te_names = []

        if fold == 1:
            final_feature_columns = X_tr.columns.tolist()
            pd.DataFrame({"feature": final_feature_columns}).to_csv(OUTDIR / "feature_columns.csv", index=False)

        print("#"*16)
        print(f"Trial {trial.number if trial is not None else 'manual'} | Fold {fold}/{CFG.FOLDS}")
        print("params:", {k: params[k] for k in ["lr", "wd", "p_drop", "n_ens", "n_epochs"]})
        print("#"*16)

        seed_everything(CFG.SEED)
        model = RealMLP_TD_Classifier(**params)
        model.fit(X_tr, y_tr, X_val, y_val)

        val_preds = model.predict_proba(X_val)[:, 1].astype(np.float32)
        oof_preds[val_idx] = val_preds

        fold_score = roc_auc_score(y_val, val_preds)
        fold_scores.append(float(fold_score))
        fold_details.append({
            "fold": fold,
            "auc": float(fold_score),
            "n_train_comp": int(len(tr_idx)),
            "n_train_orig": int(len(or_tr_idx)),
            "n_valid": int(len(val_idx)),
        })
        print(f"Fold {fold} AUC: {fold_score:.8f}")

        if trial is not None:
            trial.set_user_attr(f"fold{fold}_auc", float(fold_score))
            # Report mean-to-date. RealMLP is expensive, so pruning is intentionally not used by default.
            trial.report(float(np.mean(fold_scores)), step=fold)

    cv_auc = float(roc_auc_score(y, oof_preds))
    print(f"Overall CV AUC: {cv_auc:.10f}")
    return cv_auc, fold_scores, fold_details, final_feature_columns


def objective(trial):
    params = make_trial_params(trial)
    cv_auc, fold_scores, fold_details, _ = run_cv_for_params(params, trial=trial)
    trial.set_user_attr("fold_scores", fold_scores)
    trial.set_user_attr("fold_details", fold_details)
    trial.set_user_attr("effective_params", params)
    return cv_auc

sampler = optuna.samplers.TPESampler(seed=CFG.OPTUNA_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler, study_name=EXP_ID)
study.optimize(objective, n_trials=CFG.N_TRIALS, timeout=CFG.TIMEOUT_SEC, gc_after_trial=True)

print("Best value:", study.best_value)
print("Best params:", study.best_params)


[I 2026-05-25 04:20:47,254] A new study created in memory with name: exp_20260525_053_realmlp_watchlist_updated_optuna_search_min


################
Trial 0 | Fold 1/5
params: {'lr': 0.01603056612222376, 'wd': 0.02284096824419205, 'p_drop': 0.06659969709057026, 'n_ens': 8, 'n_epochs': 5}
################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_RaceProgress_cat_', 'TyreL

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/5: val 1-auc_ovr = 0.057253
Epoch 2/5: val 1-auc_ovr = 0.048395
Epoch 3/5: val 1-auc_ovr = 0.047443
Epoch 4/5: val 1-auc_ovr = 0.045781
Epoch 5/5: val 1-auc_ovr = 0.045183


`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


...

Fold 4 AUC: 0.95283296
################
Trial 19 | Fold 5/5
params: {'lr': 0.018404750770168805, 'wd': 0.01916246653891711, 'p_drop': 0.05922492800757212, 'n_ens': 8, 'n_epochs': 5}
################
Columns classified as continuous: ['Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', '_LapNumber_/_RaceProgress', '_TyreLife_/_LapNumber', '_LapTime (s)_*_Cumulative_Degradation', '_LapTime (s)_*_Cumulative_Degradation_abs', '_LapTime (s)_/_Cumulative_Degradation_abs', '_Driver_count', '_Compound_count', '_Race_count', '_Year_cat_count', '_PitStop_cat_count', '_Race_Compound_TE', '_Race_Year_TE']
Columns classified as categorical: ['Driver', 'Compound', 'Race', 'Year_cat_', 'PitStop_cat_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_', 'LapNumber_/_R

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/5: val 1-auc_ovr = 0.056515
Epoch 2/5: val 1-auc_ovr = 0.048187
Epoch 3/5: val 1-auc_ovr = 0.047104
Epoch 4/5: val 1-auc_ovr = 0.046015
Epoch 5/5: val 1-auc_ovr = 0.045330


`Trainer.fit` stopped: `max_epochs=5` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
[I 2026-05-25 07:33:37,301] Trial 19 finished with value: 0.9537520811535959 and parameters: {'lr': 0.018404750770168805, 'wd': 0.01916246653891711, 'p_drop': 0.05922492800757212}. Best is trial 19 with value: 0.9537520811535959.


Fold 5 AUC: 0.95466970
Overall CV AUC: 0.9537520812
Best value: 0.9537520811535959
Best params: {'lr': 0.018404750770168805, 'wd': 0.01916246653891711, 'p_drop': 0.05922492800757212}
CPU times: user 3h 10min 31s, sys: 2min 49s, total: 3h 13min 20s
Wall time: 3h 12min 50s


## Save Optuna Outputs


In [8]:
# Save Optuna artifacts
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state", "datetime_start", "datetime_complete", "duration"))
trials_path = OUTDIR / "optuna_trials_053.csv"
trials_df.to_csv(trials_path, index=False)

best_params = dict(study.best_params)
best_effective_params = BASE_PARAMS.copy()
best_effective_params.update(best_params)

best_params_payload = {
    "experiment_id": EXP_ID,
    "best_trial_number": int(study.best_trial.number),
    "best_cv_auc": float(study.best_value),
    "best_params": best_params,
    "effective_params_search": best_effective_params,
    "note_for_054": {
        "use_best_params": best_params,
        "refit_n_ens": 20,
        "refit_n_epochs": 5,
        "fixed_feature_set": "same_as_051",
    },
}

best_params_path = OUTDIR / "best_params_053.json"
with open(best_params_path, "w") as f:
    json.dump(best_params_payload, f, indent=2)

summary_payload = {
    "experiment_id": EXP_ID,
    "direction": "maximize",
    "n_trials_requested": CFG.N_TRIALS,
    "timeout_sec": CFG.TIMEOUT_SEC,
    "n_trials_completed": len(study.trials),
    "best_trial_number": int(study.best_trial.number),
    "best_cv_auc": float(study.best_value),
    "best_params": best_params,
    "baseline_051_cv_auc": REFERENCE["baseline_051_cv_auc"],
    "delta_vs_051_cv_auc": float(study.best_value - REFERENCE["baseline_051_cv_auc"]),
    "search_runtime_params": {
        "n_ens": CFG.SEARCH_N_ENS,
        "n_epochs": CFG.SEARCH_N_EPOCHS,
        "folds": CFG.FOLDS,
    },
}
summary_path = OUTDIR / "optuna_study_summary_053.json"
with open(summary_path, "w") as f:
    json.dump(summary_payload, f, indent=2)

study_pkl_path = OUTDIR / "optuna_study_053.pkl"
with open(study_pkl_path, "wb") as f:
    pickle.dump(study, f)

print("Saved:")
print(" trials      :", trials_path)
print(" best params :", best_params_path)
print(" summary     :", summary_path)
print(" study pkl   :", study_pkl_path)
print("Best CV AUC  :", study.best_value)
print("Best params  :", best_params)


Saved:
 trials      : /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/optuna_trials_053.csv
 best params : /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/best_params_053.json
 summary     : /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/optuna_study_summary_053.json
 study pkl   : /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/optuna_study_053.pkl
Best CV AUC  : 0.9537520811535959
Best params  : {'lr': 0.018404750770168805, 'wd': 0.01916246653891711, 'p_drop': 0.05922492800757212}


In [9]:
# Save cv_result.json and memo_draft.yml

def to_builtin(obj):
    if isinstance(obj, dict):
        return {str(k): to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

completed_trials = [t for t in study.trials if t.value is not None]
best_trial = study.best_trial

cv_result = {
    "experiment": {
        "id": EXP_ID,
        "competition": COMPETITION,
        "metric": METRIC,
        "created_at": CREATED_AT,
        "type": "optuna_search",
        "model_family": "RealMLP / PyTabKit",
    },
    "objective": {
        "summary": "Narrow Optuna search for 051 RealMLP. Search only lr/wd/p_drop. n_epochs fixed at 5.",
        "base_experiment": "exp_20260525_051_realmlp_watchlist_updated_shared001v2_min",
        "next_refit_experiment": "exp_20260525_054_realmlp_watchlist_updated_optuna_refit_min",
    },
    "fixed": {
        "feature_set": "same_as_051",
        "folds": CFG.FOLDS,
        "seed": CFG.SEED,
        "target_encoding": CFG.TE,
        "n_epochs": CFG.SEARCH_N_EPOCHS,
        "n_ens_search": CFG.SEARCH_N_ENS,
        "forbidden": [
            "Normalized_TyreLife",
            "Normalized_TyreLife proxy",
            "future endpoint proxy",
            "pseudo label",
            "new feature engineering",
        ],
    },
    "search_space": SEARCH_SPACE,
    "base_params_search": to_builtin(BASE_PARAMS),
    "optuna": {
        "direction": "maximize",
        "n_trials_requested": CFG.N_TRIALS,
        "timeout_sec": CFG.TIMEOUT_SEC,
        "sampler": "TPESampler",
        "seed": CFG.OPTUNA_SEED,
        "n_trials_completed": len(study.trials),
        "n_completed_with_value": len(completed_trials),
        "best_trial_number": int(best_trial.number),
        "best_cv_auc": float(study.best_value),
        "best_params": to_builtin(study.best_params),
        "best_effective_params_search": to_builtin(best_effective_params),
        "best_user_attrs": to_builtin(best_trial.user_attrs),
    },
    "baselines": REFERENCE,
    "comparison": {
        "delta_best_cv_vs_051": float(study.best_value - REFERENCE["baseline_051_cv_auc"]),
        "need_054_refit": True,
        "note": "053 is search-stage only. Use 054 formal refit before blend/adoption judgement.",
    },
    "artifacts": {
        "outdir": str(OUTDIR),
        "best_params": str(best_params_path),
        "trials": str(trials_path),
        "study_summary": str(summary_path),
        "study_pkl": str(study_pkl_path),
        "feature_columns_base_before_te": str(OUTDIR / "feature_columns_base_before_te.csv"),
        "feature_columns": str(OUTDIR / "feature_columns.csv"),
    },
}

cv_result_path = OUTDIR / "cv_result.json"
with open(cv_result_path, "w") as f:
    json.dump(to_builtin(cv_result), f, indent=2)

memo = f"""
experiment:
  id: {EXP_ID}
  competition: {COMPETITION}
  metric: {METRIC}
  created_at: {CREATED_AT}
  type: optuna_search
  status: completed

objective:
  summary: >
    051 watchlist updated RealMLPの特徴量・fold・original concatを固定し、
    RealMLPのlr / wd / p_dropのみを狭くOptuna探索する。
    n_epochsは5で固定し、探索段階ではn_ens=8に軽量化する。
    目的は054 refit候補のbest paramsを取得すること。

base:
  experiment: exp_20260525_051_realmlp_watchlist_updated_shared001v2_min
  model: RealMLP_TD_Classifier
  baseline_051:
    cv_auc: {REFERENCE['baseline_051_cv_auc']}
    public_lb: {REFERENCE['baseline_051_public_lb']}

fixed:
  feature_set: same_as_051
  folds: {CFG.FOLDS}
  seed: {CFG.SEED}
  target_encoding: {CFG.TE}
  n_epochs: {CFG.SEARCH_N_EPOCHS}
  n_ens_search: {CFG.SEARCH_N_ENS}
  forbidden:
    - Normalized_TyreLife
    - Normalized_TyreLife proxy
    - future endpoint proxy
    - pseudo label
    - new feature engineering

search_space:
  lr:
    type: float_log
    low: {SEARCH_SPACE['lr']['low']}
    high: {SEARCH_SPACE['lr']['high']}
  wd:
    type: float_log
    low: {SEARCH_SPACE['wd']['low']}
    high: {SEARCH_SPACE['wd']['high']}
  p_drop:
    type: float
    low: {SEARCH_SPACE['p_drop']['low']}
    high: {SEARCH_SPACE['p_drop']['high']}

optuna:
  direction: maximize
  n_trials_requested: {CFG.N_TRIALS}
  timeout_sec: {CFG.TIMEOUT_SEC}
  sampler: TPESampler
  seed: {CFG.OPTUNA_SEED}
  n_trials_completed: {len(study.trials)}
  best_trial_number: {int(best_trial.number)}
  best_cv_auc: {float(study.best_value)}
  best_params: {to_builtin(study.best_params)}
  delta_best_cv_vs_051: {float(study.best_value - REFERENCE['baseline_051_cv_auc'])}

outputs:
  best_params: best_params_053.json
  trials: optuna_trials_053.csv
  study_summary: optuna_study_summary_053.json
  study_pkl: optuna_study_053.pkl
  cv_result: cv_result.json
  memo_draft: memo_draft.yml

next_step:
  - >
    054でbest_params_053.jsonを読み込み、051構成へ適用して正式OOF/PRED/submissionを作る。
  - >
    054ではn_ens=20、n_epochs=5に戻す。
  - >
    054単体CV/LBとAdd054 blendで、052 pruned NM/LogRegを上回るか確認する。

success_criteria_for_054:
  single:
    cv_must_exceed_051: {REFERENCE['baseline_051_cv_auc']}
    public_lb_must_be_at_least: {REFERENCE['baseline_051_public_lb']}
  blend:
    nm_cv_must_exceed: {REFERENCE['current_best_attack_052_nm_cv_auc']}
    nm_public_lb_must_be_at_least: {REFERENCE['current_best_attack_052_nm_public_lb']}
    logreg_public_lb_must_be_at_least: {REFERENCE['current_best_stable_052_logreg_public_lb']}

judgement:
  decision: pending_054_refit
  note: >
    053は探索段階であり、正式採用判断はしない。
    054 refitおよびAdd054 blend後に判断する。
""".strip()

memo_path = OUTDIR / "memo_draft.yml"
with open(memo_path, "w") as f:
    f.write(memo + "\n")

print("Saved:")
print(" cv_result :", cv_result_path)
print(" memo      :", memo_path)


Saved:
 cv_result : /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/cv_result.json
 memo      : /kaggle/working/exp_20260525_053_realmlp_watchlist_updated_optuna_search_min/memo_draft.yml
